# 4.2 — Calculate the eight-method runtime comparison

Run this notebook in **Google Colab with a standard CPU runtime**. Calculations run on
**your own Modal account**, not on Colab. No local GC2D installation or B2 account is needed.
The author's public B2 bucket supplies verified, frozen inputs and executable sources.
New results are offered as a **ZIP download to your computer**, including all eight
trajectories, individual timings, parameters, checksums and source provenance.

Create a [Modal account](https://modal.com/) and a personal API token in your Modal settings.
The [Starter offer](https://modal.com/pricing) currently includes **30 USD of compute credit
per month** (checked 27 November 2026); resource use beyond the included allowance can be billed.
Installation, authentication and calculation settings are configured below.

The reference experiment uses 40 particles, reflected potential, normalized rho=0.015,
40 fixed steps and ten saved samples per cycle, float64, 500 cycles and eight methods.
One first call plus three cached repetitions per method means **32 integrations**.
All methods run sequentially in one two-core Modal CPU container (4 GiB requested, 8 GiB limit).
Timing includes preparation, integration, synchronization and host results; it excludes
container startup, input decoding, audits, transfers and file saving. First calls include
compilation. The original sources and dependencies are frozen to reproduce the protocol;
execution times can differ with the available hardware.

To view the reference figure without calculating, open the
[visualization notebook](https://colab.research.google.com/drive/1_AFGcTjOxlplLJdRxEfFquAkIQ_LiToS).
Run the cells in order. Calculation is enabled explicitly in the settings cell.


## 1. Install the Modal client
The numerical libraries are installed in the remote container, not in Colab.


In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    installed = importlib.metadata.version("modal")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "1.6.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "modal==1.6.0"], check=True)
import modal
print("Modal client ready.")


## 2. Authenticate with your personal Modal account

In Colab's **Secrets** panel (key icon), add `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET`
and grant this notebook access. Alternatively, enter them in the hidden prompts below.
Do not paste credentials into code cells. They are held in the runtime environment;
they are never added to the downloaded results or saved notebook outputs.
Existing environment credentials are reused when running locally.


In [ ]:
import getpass
import os

for key in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"):
    if not os.environ.get(key):
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(key)
        except Exception:
            pass
        if not value:
            value = getpass.getpass(f"{key}: ")
        if not value:
            raise ValueError("Both personal Modal credentials are required.")
        os.environ[key] = value.strip()
        del value
print("Credentials loaded for this runtime; no calculation has started.")


## 3. Choose the calculation

Keep **500 cycles and 3 repetitions** to reproduce the reference protocol. A shorter
duration such as 1 cycle checks the workflow but does not reproduce the thesis timings.
Keep the generated run identifier to recover the same calculation. Choose a new identifier
before starting another experiment. Set `LAUNCH_CALCULATION = True` when ready.

After a disconnected Colab session, enter the previous `RUN_ID`, settings and
`RESUME_CALL_ID` from the receipt to recover that call without repeating the calculation.
Modal call results have limited retention; download the ZIP promptly. The `.ipynb` file
itself does not contain the result archive, and Colab runtime files are temporary.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

CYCLES = 500  # @param {type:"integer"}
REPETITIONS = 3  # @param {type:"integer"}
LAUNCH_CALCULATION = False  # @param {type:"boolean"}
RUN_ID = globals().get("RUN_ID", "reader-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-") + uuid4().hex[:6])
RESUME_CALL_ID = ""  # @param {type:"string"}
WORKSPACE = Path.cwd() / "gc2d-colab-42"
WORKSPACE.mkdir(exist_ok=True)
print(f"Run: {RUN_ID}; {CYCLES} cycles; {8 * (REPETITIONS + 1)} integrations.")


## 4. Load the verified reproduction kit

This downloads about 3.8 MB from public B2. Checksums are verified before the archived
Python inputs are used. The helper code below is embedded so no repository checkout is
needed; its reviewable sources accompany the original notebook.


In [ ]:
import importlib
import json
import sys

KIT_URL = 'https://f003.backblazeb2.com/file/gc2d-colab-public/04_02_cout_calcul/calculation/v1/calculation-kit.tar.xz'
KIT_SHA256 = 'c7eac5a241a834bf910ca7cd910a46918e4c7168eebd678a496a19121d888375'
(WORKSPACE / "calculation_runtime.py").write_text('"""Portable, standard-library-only preparation, recovery and result checks."""\nfrom hashlib import sha256\nimport io\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport tarfile\nfrom urllib.request import urlopen\nimport zipfile\n\nMETHODS = (\'RK2\', \'ImplicitMidpoint\', \'RK4\', \'GaussLegendre4\',\n    \'ABBA2Midpoint\', \'ABBA2Implicit\', \'BM4Midpoint\', \'BM4Implicit\')\n\n\ndef check_settings(cycles, repetitions, run_id):\n    if type(cycles) is not int or not 1 <= cycles <= 500:\n        raise ValueError(\'Choose an integer duration from 1 to 500 cycles.\')\n    if type(repetitions) is not int or not 3 <= repetitions <= 10:\n        raise ValueError(\'Choose 3 to 10 cached repetitions.\')\n    if not isinstance(run_id, str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_-]{0,79}\', run_id):\n        raise ValueError(\'Use a short run identifier containing letters, numbers, - or _.\')\n    return dict(cycles=cycles, repetitions=repetitions, run_id=run_id)\n\n\ndef unpack_tar(data, destination):\n    """Reject traversal, links and duplicates before writing any trusted bundle."""\n    with tarfile.open(fileobj=io.BytesIO(data), mode=\'r:*\') as archive:\n        members = archive.getmembers()\n        seen = set()\n        for member in members:\n            path = PurePosixPath(member.name)\n            if (not member.isfile() or path.is_absolute() or \'..\' in path.parts\n                    or \'\\\\\' in member.name or member.name in seen):\n                raise ValueError(\'Unsafe or duplicated archive entry.\')\n            target = destination / member.name\n            if target.resolve().is_relative_to(destination.resolve()) is False or target.is_symlink():\n                raise ValueError(\'Unsafe extraction destination.\')\n            seen.add(member.name)\n        for member in members:\n            target = destination / member.name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(archive.extractfile(member).read())\n\n\ndef verify_inputs(kit):\n    manifest = json.loads((kit / \'inputs.json\').read_text())\n    expected = {\'source.tar.gz\', \'study.json\'} | {m + s for m in METHODS for s in (\'.json\', \'.payload\')}\n    if set(manifest[\'files\']) != expected:\n        raise ValueError(\'Unexpected reproduction-kit contents.\')\n    for name, entry in manifest[\'files\'].items():\n        data = (kit / name).read_bytes()\n        if len(data) != entry[\'bytes\'] or sha256(data).hexdigest() != entry[\'sha256\']:\n            raise ValueError(f\'Damaged input: {name}\')\n    return manifest\n\n\ndef prepare_kit(data, expected_sha256, destination):\n    if sha256(data).hexdigest() != expected_sha256:\n        raise ValueError(\'Reproduction-kit checksum mismatch; nothing was executed.\')\n    unpack_tar(data, destination)\n    manifest = verify_inputs(destination)\n    unpack_tar((destination / \'source.tar.gz\').read_bytes(), destination / \'source\')\n    return manifest\n\n\ndef download_kit(url, expected_sha256, destination):\n    with urlopen(url, timeout=120) as response:\n        data = response.read()\n    return prepare_kit(data, expected_sha256, destination)\n\n\ndef write_json(path, record):\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(record, indent=2, sort_keys=True) + \'\\n\')\n    temporary.replace(path)\n\n\ndef launch_or_recover(worker, receipt_path, settings, kit_sha256, resume_call_id=\'\'):\n    """A pending receipt prevents duplicate jobs after an ambiguous submission."""\n    import modal\n    requested = {**settings, \'kit_sha256\': kit_sha256}\n    if receipt_path.exists():\n        receipt = json.loads(receipt_path.read_text())\n        if receipt[\'settings\'] != requested:\n            raise ValueError(\'This run identifier has different settings; choose a new one.\')\n        if receipt[\'submission\'] != \'confirmed\':\n            raise RuntimeError(\'Submission was interrupted. Inspect your Modal dashboard before starting another run.\')\n        if resume_call_id and resume_call_id != receipt[\'call_id\']:\n            raise ValueError(\'The saved call differs from RESUME_CALL_ID.\')\n        return receipt\n    receipt_path.parent.mkdir(parents=True, exist_ok=True)\n    if resume_call_id:\n        receipt = dict(submission=\'confirmed\', call_id=resume_call_id, settings=requested)\n        write_json(receipt_path, receipt)\n        return receipt\n    with modal.enable_output(), worker.app.run(detach=True):\n        write_json(receipt_path, dict(submission=\'unconfirmed\', settings=requested))\n        call = worker.calculate.spawn(**settings)\n        receipt = dict(submission=\'confirmed\', call_id=call.object_id, settings=requested)\n        write_json(receipt_path, receipt)\n    return receipt\n\n\ndef verify_results(data, settings):\n    """Check the returned archive before saving or offering it for download."""\n    with zipfile.ZipFile(io.BytesIO(data)) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)) or any(PurePosixPath(n).is_absolute() or \'..\' in PurePosixPath(n).parts for n in names):\n            raise ValueError(\'Unsafe result archive.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if set(names) != set(manifest[\'files\']) | {\'manifest.json\'}:\n            raise ValueError(\'Result manifest does not cover the complete ZIP.\')\n        for name, entry in manifest[\'files\'].items():\n            value = archive.read(name)\n            if len(value) != entry[\'bytes\'] or sha256(value).hexdigest() != entry[\'sha256\']:\n                raise ValueError(f\'Damaged result: {name}\')\n        study = json.loads(archive.read(\'study.json\'))\n        if any(study[k] != v for k, v in settings.items()):\n            raise ValueError(\'The recovered call used different settings.\')\n        if study[\'methods\'] != list(METHODS) or study[\'particle_count\'] != 40:\n            raise ValueError(\'Wrong methods or particles.\')\n        timings = json.loads(archive.read(\'timings.json\'))\n        expected = {(m, r) for m in METHODS for r in range(settings[\'repetitions\'] + 1)}\n        rows = timings[\'measurements\']\n        if len(rows) != len(expected) or {(r[\'method\'], r[\'repetition\']) for r in rows} != expected:\n            raise ValueError(\'Incomplete or duplicated timing measurements.\')\n        if not timings[\'same_container\'] or not timings[\'repeated_states_identical\']:\n            raise ValueError(\'Worker audits failed.\')\n        for row in rows:\n            if row[\'step_count\'] != settings[\'cycles\'] * 40:\n                raise ValueError(\'Incorrect step count.\')\n            first = row[\'repetition\'] == 0\n            if (row[\'integration_cache_misses\'], row[\'integration_cache_hits\']) != ((1, 0) if first else (0, 1)):\n                raise ValueError(\'Invalid compilation-cache audit.\')\n            order = METHODS if row[\'repetition\'] % 2 == 0 else METHODS[::-1]\n            if row[\'position\'] != order.index(row[\'method\']) or row[\'phase\'] != (\'first\' if first else \'warm\'):\n                raise ValueError(\'Incorrect repetition order or phase.\')\n            if not math.isfinite(row[\'seconds\']) or row[\'seconds\'] <= 0:\n                raise ValueError(\'Invalid elapsed time.\')\n        for method in METHODS:\n            if not any(n.startswith(f\'runs/{method}/\') and n.endswith(\'.npz\') for n in names):\n                raise ValueError(\'A trajectory archive is missing.\')\n        return study, timings\n')
(WORKSPACE / "calculation_worker.py").write_text('"""Importable Modal worker: numerical dependencies exist only in its container."""\nfrom hashlib import sha256\nimport io\nimport json\nfrom pathlib import Path\nimport tempfile\nimport zipfile\n\nimport modal\n\nROOT = Path(__file__).resolve().parent\nimage = (modal.Image.debian_slim(python_version=\'3.12\')\n    .pip_install(\'numpy==1.26.4\', \'scipy==1.17.1\', \'h5py==3.16.0\',\n        \'threadpoolctl==3.6.0\', \'jax==0.7.1\', \'jaxlib==0.7.1\')\n    .env({\'PYTHONPATH\': \'/opt/gc2d/source/src\', \'JAX_ENABLE_X64\': \'true\',\n        \'JAX_PLATFORMS\': \'cpu\', \'JAX_ENABLE_COMPILATION_CACHE\': \'false\',\n        \'OMP_NUM_THREADS\': \'2\', \'OPENBLAS_NUM_THREADS\': \'2\', \'MKL_NUM_THREADS\': \'2\'})\n    .add_local_dir(ROOT / \'kit\', \'/opt/gc2d\')\n    .add_local_python_source(\'calculation_runtime\'))\napp = modal.App(\'gc2d-colab-method-timings\')\n\n\n@app.function(image=image, cpu=(2., 2.), memory=(4096, 8192), timeout=14400,\n    retries=0, max_containers=1, scaledown_window=60)\ndef calculate(cycles=500, repetitions=3, run_id=\'reader-run\'):\n    """Run the frozen benchmark, then return portable result files as ZIP bytes."""\n    from calculation_runtime import check_settings, verify_inputs\n    from diagnostics.io.persistence import save_solution, load_solution\n    from execution._modal_worker import decode_job, encode_job\n    from request import SimulationRequest\n    from studies.method_timings import benchmark_methods, METHODS\n    import numpy as np\n\n    settings = check_settings(cycles, repetitions, run_id)\n    kit = Path(\'/opt/gc2d\')\n    inputs = verify_inputs(kit)\n    payloads = {name: (kit / (name + \'.payload\')).read_bytes() for name in METHODS}\n    if cycles != 500:\n        # Short checks reuse exactly the physical model, methods and fixed step.\n        schedule = SimulationRequest.uniform(t_span=(0., float(cycles)), sample_count=cycles * 10 + 1)\n        for name, payload in payloads.items():\n            problem, method, options, observers = decode_job(payload)\n            payloads[name] = encode_job(problem.with_request(schedule), method, options, observers)\n    result = benchmark_methods(payloads, repetitions)\n    solutions = result.pop(\'solutions\')\n    record = json.loads((kit / \'study.json\').read_text())\n    record.pop(\'bucket\', None)\n    record.update(settings, reference_run_id=inputs[\'reference_run_id\'],\n        storage=\'ZIP download to the reader computer\', reference_configuration=cycles == 500 and repetitions == 3,\n        original_input_files=inputs[\'files\'])\n    with tempfile.TemporaryDirectory(prefix=\'gc2d-results-\') as directory:\n        output = Path(directory)\n        for name, solution in solutions.items():\n            if solution.states.shape != (cycles * 10 + 1, 80) or solution.diagnostics[\'step_count\'] != cycles * 40:\n                raise ValueError(\'Incomplete trajectory.\')\n            metadata = json.loads((kit / (name + \'.json\')).read_text())\n            metadata[\'config\'][\'cycles\'] = cycles\n            metadata.update(repetitions=repetitions, run_id=run_id, payload_sha256=sha256(payloads[name]).hexdigest(),\n                environment=result[\'environment\'], original_payload_sha256=inputs[\'files\'][name + \'.payload\'][\'sha256\'])\n            save_solution(solution, output / \'runs\' / name, metadata=metadata)\n            restored = load_solution(output / \'runs\' / name)\n            np.testing.assert_array_equal(restored.solution.states, solution.states)\n        for filename, value in [(\'study.json\', record), (\'timings.json\', result), (\'inputs.json\', inputs)]:\n            (output / filename).write_text(json.dumps(value, indent=2) + \'\\n\')\n        (output / \'source.tar.gz\').write_bytes((kit / \'source.tar.gz\').read_bytes())\n        files = {p.relative_to(output).as_posix(): p.read_bytes() for p in sorted(output.rglob(\'*\')) if p.is_file()}\n        manifest = {\'schema_version\': 1, \'files\': {name: dict(bytes=len(data), sha256=sha256(data).hexdigest())\n            for name, data in files.items()}}\n        files[\'manifest.json\'] = (json.dumps(manifest, indent=2, sort_keys=True) + \'\\n\').encode()\n        buffer = io.BytesIO()\n        with zipfile.ZipFile(buffer, \'w\', zipfile.ZIP_DEFLATED) as archive:\n            for name, data in files.items():\n                archive.writestr(name, data)\n        return buffer.getvalue()\n')
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
importlib.invalidate_caches()
import calculation_runtime as runtime
runtime = importlib.reload(runtime)
settings = runtime.check_settings(CYCLES, REPETITIONS, RUN_ID)
inputs = runtime.download_kit(KIT_URL, KIT_SHA256, WORKSPACE / "kit")
import calculation_worker as worker
print("Original inputs and sources verified. No calculation has started.")


## 5. Launch once, or recover the recorded call

Keep a copy of the receipt shown below. If submission is interrupted before a call ID
is recorded, inspect your [Modal dashboard](https://modal.com/apps) before launching again.
A confirmed call is reused even if this cell is executed again.


In [ ]:
from IPython.display import display, FileLink

if not LAUNCH_CALCULATION and not RESUME_CALL_ID:
    raise RuntimeError("Set LAUNCH_CALCULATION=True in cell 3, then run cells 3 to 7.")
RUN_DIRECTORY = WORKSPACE / "results" / RUN_ID
RECEIPT = RUN_DIRECTORY / "receipt.json"
receipt = runtime.launch_or_recover(worker, RECEIPT, settings, KIT_SHA256, RESUME_CALL_ID)
print(json.dumps(receipt, indent=2))
display(FileLink(str(RECEIPT)))
try:
    from google.colab import files
    files.download(str(RECEIPT))
except ImportError:
    pass


## 6. Retrieve and verify the completed results

Wait for the recorded call. Rerunning this cell retrieves the same result; it never
launches a calculation. A timeout can be retried with the same receipt. If Modal reports
an expired result, it cannot be recovered from the notebook: check any ZIP already downloaded.
The archive contains `study.json`, `timings.json`, `runs/<method>/` with numerical arrays
and metadata, frozen sources, input fingerprints and a checksum manifest.


In [ ]:
from statistics import median

RESULT_ZIP = RUN_DIRECTORY / (RUN_ID + ".zip")
if RESULT_ZIP.exists():
    result_bytes = RESULT_ZIP.read_bytes()
else:
    result_bytes = modal.FunctionCall.from_id(receipt["call_id"]).get(timeout=3600)
study, timings = runtime.verify_results(result_bytes, settings)
if study["original_input_files"] != inputs["files"]:
    raise ValueError("The recovered result used different original inputs.")
temporary_zip = RESULT_ZIP.with_suffix(".tmp")
temporary_zip.write_bytes(result_bytes)
temporary_zip.replace(RESULT_ZIP)
print(f"Verified {len(timings['measurements'])} integrations; archive: {len(result_bytes):,} bytes.")
print(f"{'Method':20} {'First (s)':>12} {'Median (s)':>12} {'Minimum (s)':>12} {'Maximum (s)':>12}")
for name in runtime.METHODS:
    rows = [r for r in timings["measurements"] if r["method"] == name]
    first = next(r["seconds"] for r in rows if r["repetition"] == 0)
    warm = [r["seconds"] for r in rows if r["repetition"] > 0]
    print(f"{name:20} {first:12.6f} {median(warm):12.6f} {min(warm):12.6f} {max(warm):12.6f}")


## 7. Download to your computer

Run this cell and allow the browser download. Save the ZIP and receipt on your computer
before closing the Colab runtime. Results are not uploaded to the author's B2 bucket.
No personal bucket or Modal Volume is required.


In [ ]:
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    display(FileLink(str(RESULT_ZIP)))
print("Download offered. Check that the ZIP is saved on your computer.")
